# D4.2 · Runbook tiers: fully automated, human in the loop, manual

**Function D — The Agentic SOC → The Agentic SOC — Respond**

Builds on **[D4.1 · Remediation policy — what may be done without asking](https://spbreed.github.io/cyber-commons/lessons/D4.1.html)**.

| | |
|---|---|
| Tools used | kagent |

## What this lesson is

**What it covers.** The three runbook tiers — fully automated, human in the loop, manual — timed against one incident, with the cost of acting on a bad signal beside the time to contain.

**Why a security engineer needs it.** "Automate everything" and "keep a human in it" are both asserted constantly and neither is a position. The tiers trade time against the cost of being wrong, and the right trade depends on the detection's measured false-positive rate. Against a 29-minute breakout time, a manual tier that contains in 34 minutes is a risk decision — just an unstated one.

## 1 · The hook

Fully automated contains in fourteen seconds and is wrong eight times in a hundred. Manual is almost never wrong and takes thirty-four minutes, against a breakout time of twenty-nine. Neither of those is the safe option; they fail differently.

> **At CyberTravels.** The incident is CyberTravels' runaway Workflow Agent, and the numbers are its containment ladder from A3.9 timed three ways. Manual takes thirty-four minutes against a breakout time of twenty-nine, which for CyberTravels means the refunds have already moved before anybody has decided anything.

## 2 · The framework

```
   one incident, three tiers

   tier                 decide      contain      wrong / 100
   automated                1s          14s              8.0
   human in the loop      240s         253s              1.2
   manual                1800s        2072s              0.2
                                        ^
                        breakout time is 29 min = 1740s
                        manual contains AFTER the attacker finished

   deliberately no fourth column ranking these: seconds of exposure and
   wrongly-contained agents are different units, and any single score
   has an exchange rate hidden in it that somebody chose
```

The three tiers are not levels of ambition. They are three different trades
between **time to contain** and **the cost of acting on a bad signal** — and
which trade is right depends on the detection's measured false-positive rate,
not on taste.

There is no single score that ranks them, and this lesson refuses to print one.
Seconds of exposure and wrongly-contained agents are in different units. Any
number that ranks them has an exchange rate hidden inside it, chosen by whoever
wrote the formula.

## 3 · Both tiers fail; they fail differently

Fully automated contains in fourteen seconds and is wrong eight times in a
hundred, with nobody between the mistake and the estate.

Manual is wrong far less often and takes thirty-four minutes — which, against a
29-minute breakout time, means containment lands after the attacker has
finished. "Keep a human in it" is a risk decision too. It is just usually an
unstated one.

## 4 · One incident, three tiers

Two columns, deliberately unranked. Choose the exchange rate openly, per incident class, or do not choose it.

### The skill — [`skills/response/runbook-tier-assignment/SKILL.md`](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/response/runbook-tier-assignment/SKILL.md)

```yaml
name: runbook-tier-assignment
description: >-
  Run one incident's response at fully automated, human-in-the-loop and manual
  tiers, and compare time to contain against wrong actions. Use when choosing a
  runbook's tier, when justifying automation to a risk function, or when
  "automate everything" and "keep a human in it" are both being asserted.
allowed-tools: Read, Grep, Glob
```

# Three tiers, three ways of being wrong

The tiers are not levels of ambition. They are three different trades between
**time to contain** and **the cost of acting on a bad signal** — and which trade
is right depends on the detection's false-positive rate, not on taste.

There is no single score that ranks them. Seconds of exposure and wrongly
contained agents are in different units; any number that ranks them has an
exchange rate hidden inside it. Choose that rate openly, per incident class, or
do not choose it.

## When to use this

When assigning a tier to a new runbook, and when reviewing an existing one after
its detection's false-positive rate has been measured. The tier should move when
the rate moves.

## Step-by-step

**1 — Take the tier from `remediation-policy-check` as the ceiling.** This skill
chooses within what policy permits, never above it.

**2 — Time each step at each tier honestly.** The decide step is where the tiers
actually differ; the mechanical steps barely move.

**3 — Bring the detection's measured false-positive rate.** Without it this is
an argument about feelings.

**4 — Report both costs side by side, unranked.** Resist the composite score.

**5 — Choose, and write down the exchange rate you used.** The next person needs
to know what you traded, not just what you picked.

## Example

**Input** — one incident, five steps, three tiers, in
[`scripts/runbook_tier_assignment.py`](scripts/runbook_tier_assignment.py).

**Output** — a real run:

```
tier                   contain   reaches the estate wrongly / 100
automated                  14s                                8.0
human-in-the-loop         253s                                1.2
manual                   2072s                                0.2
```

Manual contains after 34 minutes, against a 29-minute breakout time.

## Output contract

```json
{
  "tiers": [{"tier": "str", "seconds_to_contain": 0, "wrong_actions_per_100": 0.0}]
}
```

## Common edge cases

- **The detection has no measured rate.** Then no tier can be justified; measure
  first.
- **The human is not actually available.** A human-in-the-loop tier with a
  four-hour on-call response is a manual tier wearing a badge.
- **The confirmation dialog.** A HITL step nobody can meaningfully refuse is an
  automated step with extra latency.

## Failure modes

- **Composite scores.** They always favour whichever axis the author weighted.
- **Automating on an unmeasured detection.** The fastest possible way to break
  production on a false positive.
- **Manual as the safe default.** On a 29-minute breakout time, slow is a risk
  decision too — just an unstated one.

In [ ]:
# The code is not in this notebook. It is this file in the repository:
#   https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/response/runbook-tier-assignment/scripts/runbook_tier_assignment.py
SCRIPT = "skills/response/runbook-tier-assignment/scripts/runbook_tier_assignment.py"
REPO = "https://github.com/spbreed/cyber-commons"
BRANCH = "claude/vulnbench-setup-scheduling-81aqov"

import glob, os, subprocess, sys

CLONE = "/kaggle/working/cyber-commons"
_root = next((r for r in (".", "..", "../..", CLONE)
              if os.path.isfile(os.path.join(r, SCRIPT))), None)

if _root is None:
    # --filter=blob:none --sparse fetches the tree without the history or the
    # notebooks; sparse-checkout then materialises only the two directories a
    # lesson needs: the procedures, and the repository they are run against.
    _c = subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none",
                         "--sparse", "--branch", BRANCH, REPO, CLONE],
                        capture_output=True, text=True)
    if _c.returncode:
        raise SystemExit(
            "could not fetch the skills: " + _c.stderr.strip()[-300:] +
            "\nOn Kaggle this needs Internet on in the notebook settings, which "
            "needs a phone-verified account. Without one, attach the dataset "
            "cybercommons/cyber-commons-skills instead — it holds the same tree.")
    # `skills` is the procedures; `cybertravels` is the sample repository they
    # scan; `curriculum` and `site/data` hold the framework mapping and the
    # session list that the reference-lookup skill reads. Miss any of them and
    # the skill clones successfully and then fails on a path that is not there,
    # which is how A0.2 failed its first Kaggle run.
    subprocess.run(["git", "-C", CLONE, "sparse-checkout", "set",
                    "skills", "cybertravels", "curriculum", "site/data"],
                   capture_output=True, text=True)
    _root = CLONE

_out = subprocess.run([sys.executable, os.path.join(_root, SCRIPT)],
                      capture_output=True, text=True,
                      env=dict(os.environ,
                               PYTHONPATH=os.path.join(_root, "skills/_runtime"),
                               PYTHONHASHSEED="0"))
print(_out.stdout, end="")
if _out.returncode:
    raise SystemExit(_out.stderr.strip()[-2000:])

## What you just proved

14s / 253s / 2072s to contain, against 8.0 / 1.2 / 0.2 wrong actions per hundred — and no third column ranking them.

## Your turn

Put your own detection's false-positive rate in and see whether the tier you already ship still looks right.

---

**Next → [D4.3 · Containment at machine speed](https://spbreed.github.io/cyber-commons/lessons/D4.3.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/D4.2.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/D4.2.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*